# ELIZA — od reguł do złudzenia rozumienia

**Podstawy AI · Laboratorium 1**

**Cel.** Zbudujesz własną wersję ELIZY — jednego z pierwszych programów
prowadzących „rozmowę" z człowiekiem — a potem spróbujesz ją *złamać*,
żeby zobaczyć, gdzie kończy się mechanizm, a zaczyna złudzenie.

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Trochę historii | czytasz (5 min) |
| 2. Wyrażenia regularne | uruchamiasz przykłady, rozgrzewka |
| 3–6. Budowa ELIZY | Zadania 1–4 |
| 7. Rangowanie reguł | Zadanie 5 ★ (dla chętnych) |
| 8. Część B — złam ELIZĘ | Zadanie 6 |
| 9. Refleksja | Zadanie 7 |

**Co oddajesz:** uzupełniony notebook — działający kod (Zadania 1–4)
oraz odpowiedzi pisemne (Zadania 6–7). Zadanie 5 ★ jest dodatkowe.

> Teza, którą będziemy sprawdzać: **wrażenie rozumienia powstaje po stronie
> człowieka, nie maszyny.**

## 1. Trochę historii

**Kto i kiedy.** ELIZĘ napisał **Joseph Weizenbaum** w MIT w latach 1964–1966.
Opis ukazał się w styczniu 1966 r. w *Communications of the ACM* pod tytułem
*„ELIZA — A Computer Program For the Study of Natural Language Communication
Between Man and Machine"*. Program działał na komputerze IBM 7094
w systemie z podziałem czasu — użytkownik pisał na dalekopisie, a odpowiedź
drukowała się po chwili.

**Skąd nazwa.** Od Elizy Doolittle z *Pigmaliona* G. B. Shawa — kwiaciarki,
którą uczy się „mówić jak dama". Program też można było „uczyć" mówić,
podmieniając mu zestaw reguł (tzw. *skrypt*).

**DOCTOR.** Najsłynniejszy skrypt udawał psychoterapeutę w nurcie
**rogeriańskim** (Carl Rogers). Taki terapeuta niewiele mówi od siebie —
głównie odbija i parafrazuje słowa pacjenta: *„Mówisz, że…?"*,
*„Jak się z tym czujesz?"*. To był sprytny wybór: w tej roli milczenie
o świecie wygląda na profesjonalizm, a nie na niewiedzę.
Program nie musiał nic wiedzieć — wystarczyło przekształcać zdania.

**Efekt ELIZY.** Ludzie bardzo szybko zaczęli przypisywać programowi
zrozumienie i empatię. Weizenbaum opisywał, że jego sekretarka — która
widziała, jak program powstaje — po kilku wymianach zdań poprosiła go,
by wyszedł z pokoju. Tę skłonność do dopatrywania się rozumienia
w prostym mechanizmie nazywamy dziś **efektem ELIZY**.

**Reakcja twórcy.** Weizenbauma to zaniepokoiło. W książce
*Computer Power and Human Reason* (1976) krytykował pomysł, by maszynom
powierzać decyzje wymagające ludzkiego osądu i troski.

**Dlaczego to wciąż ważne.** W badaniu Jonesa i Bergena (UC San Diego, 2023)
nad testem Turinga ELIZA częściej uchodziła za człowieka niż GPT-3.5 —
m.in. dlatego, że jej wymijające odpowiedzi nie wyglądały na „typowego asystenta".
Mechanizm z 1966 r. i złudzenie, które wywołuje, nadal są punktem odniesienia
w dyskusjach o dzisiejszych chatbotach.

**Jak działa (w skrócie)** — trzy operacje, nic więcej:

1. **słowo kluczowe** — znajdź w zdaniu wyzwalacz,
2. **dekompozycja** — dopasuj wzorzec z „dziurą", np. `potrzebuję (.*)`,
3. **rekompozycja** — wstaw wycięty fragment do szablonu, odwracając zaimki.

Nie ma tu pamięci, uczenia ani reprezentacji znaczenia.
Wszystkie trzy operacje zbudujemy z pomocą **wyrażeń regularnych**.

## 2. Wyrażenia regularne — tyle, ile potrzeba

**Wyrażenie regularne** (*regex*) to wzorzec opisujący, jak wygląda fragment
tekstu. W Pythonie obsługuje je moduł `re`. Wzorce zapisujemy jako
**surowe napisy** `r"..."` — wtedy `\b` czy `\w` trafia do `re` bez zmian,
a nie jest interpretowane przez Pythona jako znak specjalny.

### Ściągawka

| Wzorzec | Znaczenie | Przykład |
|---|---|---|
| `abc` | dosłownie te znaki | `czuję` pasuje do „czuję się źle" |
| `.` | dowolny jeden znak | `k.t` → kot, kit, kat |
| `*` | poprzedni element 0 lub więcej razy | `ha*` → h, ha, haaa |
| `.*` | „cokolwiek, dowolnie długie" — nasza **dziura** | |
| `?` | poprzedni element opcjonalny | `kota?` → kot, kota |
| `( )` | **grupa** — zapamiętaj, co pasowało | `potrzebuję (.*)` |
| `a\|b` | alternatywa: a **albo** b | `matka\|ojciec` |
| `\b` | granica słowa | `\bkot\b` nie trafi w „kotlet" |
| `\w` | znak słowa (litera — także polska — cyfra, `_`) | |
| `\s` | biały znak (spacja, tab, nowa linia) | |
| `[...]` | jeden znak z zestawu | `[aeiou]` |
| `[^...]` | jeden znak **spoza** zestawu | `[^\w\s]` = interpunkcja |

### Funkcje, których użyjemy

| Funkcja | Co robi |
|---|---|
| `re.search(wzorzec, tekst)` | szuka wzorca **gdziekolwiek** w tekście; zwraca obiekt dopasowania albo `None` |
| `m.group(1)` | fragment złapany przez pierwszą grupę `( )` |
| `m.groups()` | krotka wszystkich grup |
| `re.sub(wzorzec, zamiennik, tekst)` | zamienia wszystkie wystąpienia wzorca |

Uruchom kolejne komórki i przyjrzyj się wynikom.

In [ ]:
import re

m = re.search(r"potrzebuję (.*)", "chyba potrzebuję urlopu")
print(m)            # obiekt dopasowania
print(m.group(0))   # całe dopasowanie
print(m.group(1))   # to, co złapała grupa (.*)
print(m.groups())   # krotka wszystkich grup

<re.Match object; span=(6, 23), match='potrzebuję urlopu'>
potrzebuję urlopu
urlopu
('urlopu',)


In [ ]:
# Brak dopasowania -> None. Dlatego zawsze sprawdzamy `if m:`
m = re.search(r"potrzebuję (.*)", "wszystko w porządku")
print(m)
if m:
    print("trafienie")
else:
    print("brak trafienia")

None
brak trafienia


In [ ]:
# Pułapka 1: interpunkcja. Wzorzec wymaga SPACJI po słowie „potrzebuję",
# a w zdaniu jest przecinek — więc nic nie pasuje.
print(re.search(r"potrzebuję (.*)", "potrzebuję, żeby ktoś mnie słuchał"))

# Pułapka 2: (.*) łapie wszystko do końca — także kropkę.
print(re.search(r"czuję się (.*)", "czuję się samotny.").group(1))

None
samotny.


In [ ]:
# Rozwiązanie obu pułapek: przed dopasowaniem NORMALIZUJEMY tekst —
# małe litery + usunięcie interpunkcji wzorcem [^\w\s] („nie litera i nie spacja").

def normalizuj(tekst):
    tekst = tekst.lower()
    tekst = re.sub(r"[^\w\s]", "", tekst)   # usuń wszystko poza literami, cyframi i spacjami
    return tekst.strip()

print(normalizuj("Potrzebuję, żeby ktoś mnie słuchał!"))
print(re.search(r"potrzebuję (.*)", normalizuj("Potrzebuję, żeby ktoś mnie słuchał!")).group(1))

potrzebuję żeby ktoś mnie słuchał
żeby ktoś mnie słuchał


In [ ]:
# Alternatywa | i granica słowa \b
wzorzec = r"\b(matka|ojciec|rodzina)\b"
for zdanie in ["mój ojciec krzyczy", "rodzinaaa", "lubię kotlety"]:
    m = re.search(wzorzec, zdanie)
    print(f"{zdanie!r:25} ->", m.group(1) if m else None)

'mój ojciec krzyczy'      -> ojciec
'rodzinaaa'               -> None
'lubię kotlety'           -> None


### Wypełnianie szablonu: `str.format`

Odpowiedzi ELIZY przechowujemy jako **szablony** z miejscem `{0}`.
Metoda `.format()` wstawia w `{0}` pierwszy argument, w `{1}` drugi itd.

Zapis `.format(*grupy)` **rozpakowuje** listę: `grupy[0]` trafia do `{0}`,
`grupy[1]` do `{1}`. Bez gwiazdki cała lista wylądowałaby w `{0}` — razem
z nawiasami kwadratowymi.

Uwaga na numerację: pierwsza grupa regexa to `m.group(1)`, ale w krotce
`m.groups()` ma indeks 0 — i właśnie trafia do `{0}`. Wszystko się zgadza.

In [ ]:
szablon = "Dlaczego chciałbyś {0}?"
grupy = ["mieć więcej czasu"]

print(szablon.format(*grupy))   # poprawnie
print(szablon.format(grupy))    # bez * — widać nawiasy listy

# Szablon bez {0} po prostu ignoruje argumenty:
print("Opowiedz mi o tym więcej.".format(*grupy))

Dlaczego chciałbyś mieć więcej czasu?
Dlaczego chciałbyś ['mieć więcej czasu']?
Opowiedz mi o tym więcej.


### Rozgrzewka

Napisz wzorzec, który z tekstu *„od zawsze lubię grać w szachy"* wytnie
fragment po słowie *lubię*, a potem wstaw go do szablonu
`"Co najbardziej lubisz w tym, żeby {0}?"`.

In [ ]:
tekst = "od zawsze lubię grać w szachy"
wzorzec = r"________"          # <- uzupełnij
m = re.search(wzorzec, tekst)
print(m.group(1) if m else "brak dopasowania")

grać w szachy


## 3. Przygotowanie

Nie używamy `input()` jako podstawy — blokuje kernel i utrudnia sprawdzanie.
Testujemy funkcje na **liście zdań**.

Rozmowę na żywo możesz uruchomić opcjonalnie w sekcji 6.

In [ ]:
import re
import random

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]

def normalizuj(tekst):
    # małe litery, bez interpunkcji — patrz sekcja 2
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

print("Gotowe. Zdań testowych:", len(TESTY))

Gotowe. Zdań testowych: 6


## 4. Mechanizm 1 — odbicie zaimków

ELIZA odwraca perspektywę: słowa pacjenta o **sobie** stają się słowami
terapeuty o **pacjencie** — *„moja rodzina mnie słucha"* →
*„twoja rodzina ciebie słucha"*.

Działa to **w obie strony**: gdy pacjent mówi o terapeucie
(*„o co **ci** chodzi"*), terapeuta odpowiada o sobie (*„o co **mi** chodzi"*).
Bez odbicia odpowiedź brzmiałaby jak echo.

**Uwaga (wrócimy do niej w Zadaniu 6):** polszczyzna ma przypadki,
więc podstawianie słowo-za-słowo czasem da wynik niegramatyczny.
Na razie akceptujemy tę niedoskonałość.

### 🔧 Zadanie 1
1. Uzupełnij słownik `odbicia` — dodaj **co najmniej 6** par, w tym
   **co najmniej 2 w kierunku odwrotnym** (od terapeuty do pacjenta, np. `ci`).
2. Dokończ funkcję `odbij`.

*Wskazówka:* `slownik.get(klucz, domyślna)` zwraca wartość dla klucza,
a gdy klucza nie ma — wartość domyślną.

In [ ]:
odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    # TODO (Zadanie 1): dopisz min. 6 par, np. mój/twój, moje/twoje, mam/masz,
    #                   mi/ci, ja/ty ... oraz min. 2 w kierunku odwrotnym (ci -> mi ...)
}

def odbij(fragment):
    slowa = fragment.lower().split()
    # TODO (Zadanie 1): zamień każde słowo na jego odbicie, jeśli jest w słowniku,
    #                   w przeciwnym razie zostaw bez zmian.
    return " ".join(________ for s in slowa)   # <- uzupełnij

In [ ]:
# Test Zadania 1 — powinno przejść bez błędu:
assert odbij("moja rodzina") == "twoja rodzina"
assert odbij("mnie") == "ciebie"
assert odbij("o co ci chodzi") == "o co mi chodzi"   # kierunek odwrotny!
print(odbij("jestem zmęczony i moja praca mnie męczy"))
print("Zadanie 1: OK")

## 5. Mechanizmy 2 + 3 — reguły i odpowiedź

Każda reguła to para: **wzorzec** (regex) + **lista szablonów** odpowiedzi.
Reguły sprawdzamy **po kolei**; wygrywa **pierwsza**, która pasuje.

Z pasującej reguły:

1. bierzemy grupy (`m.groups()`),
2. odbijamy w nich zaimki (`odbij`),
3. losujemy szablon i wstawiamy grupy (`.format(*grupy)`).

Reguły bez `(.*)` (np. reakcja na słowo „rodzina") mają szablony bez `{0}`.

### 🔧 Zadanie 2
Dopisz **co najmniej 4** własne reguły. Sprawdź, czy ich szablony są
gramatyczne po wstawieniu fragmentu — i nie zapomnij o znaku zapytania.

*Wskazówka:* tekst jest już znormalizowany (małe litery, bez interpunkcji),
więc wzorce piszemy **małymi literami**.

### 🔧 Zadanie 3
Dokończ funkcję `odpowiedz`.

In [ ]:
# (wzorzec, [szablony]) — dopasowujemy na tekście po normalizuj()
reguly = [
    (r"potrzebuję (.*)",
        ["Dlaczego potrzebujesz {0}?", "Czy naprawdę pomogłoby ci {0}?"]),
    (r"czuję się (.*)",
        ["Od jak dawna czujesz się {0}?", "Czy często tak się czujesz?"]),
    (r"\b(matka|ojciec|rodzina|siostra|brat)\b",
        ["Opowiedz mi więcej o swojej rodzinie.", "Jak układają się te relacje?"]),
    # TODO (Zadanie 2): dodaj min. 4 własne reguły
]

In [ ]:
def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            # TODO (Zadanie 3a): odbij zaimki w każdej przechwyconej grupie
            grupy = [________ for g in m.groups()]      # <- uzupełnij
            # TODO (Zadanie 3b): wylosuj szablon i wstaw grupy w miejsce {0}, {1}...
            return ________                              # <- uzupełnij
    return None   # brak dopasowania — regułę awaryjną dodasz w Zadaniu 4

In [1]:
# Test Zadań 2+3:
for zdanie in TESTY:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n")

NameError: name 'TESTY' is not defined

## 6. Złożenie w bota + reguła awaryjna

Prawdziwa ELIZA zawsze coś odpowiada — nawet gdy nic nie pasuje.
Robi to *reguła awaryjna* (ang. *fallback*): wzorzec `r".*"`
pasuje do **każdego** tekstu.

### 🔧 Zadanie 4
1. Dopisz regułę awaryjną **na koniec** listy `reguly` (dlaczego na koniec?).
2. Uruchom rozmowę na zdaniach testowych — nie powinno być już żadnego `None`.

In [ ]:
# TODO (Zadanie 4): dopisz na końcu listy 'reguly' regułę awaryjną, np.:
# (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?", "Jak się z tym czujesz?"])
________   # <- uzupełnij (np. reguly.append(...))

def rozmowa(wejscia):
    for zdanie in wejscia:
        print(f"Pacjent: {zdanie}")
        print(f"ELIZA:   {odpowiedz(zdanie)}\n")

rozmowa(TESTY)

In [ ]:
# (Opcjonalnie) Rozmowa na żywo. Wpisz 'koniec', by wyjść.
# UWAGA: przy „Run All" ta komórka zatrzyma notebook — uruchamiaj ją ręcznie.
while True:
    tekst = input("Ty: ")
    if tekst.strip().lower() == "koniec":
        break
    print("ELIZA:", odpowiedz(tekst))

## 7. ★ Rangowanie słów kluczowych (zadanie dodatkowe)

W wersji z Zadania 3 wygrywa **pierwsza** pasująca reguła — więc o odpowiedzi
decyduje przypadkowa kolejność na liście. Prawdziwa ELIZA nadawała słowom
kluczowym **rangi** i spośród wszystkich trafień wybierała najważniejsze.

### 🔧 Zadanie 5 ★
Dokończ `odpowiedz_p`: zbierz **wszystkie** pasujące reguły i wybierz tę
o najwyższym priorytecie (fallback ma priorytet 0).

*Wskazówka:* `max(lista, key=lambda t: t[0])` zwraca element listy
o największym pierwszym polu.

In [ ]:
# (priorytet, wzorzec, [szablony]) — wyższy priorytet = ważniejsze
reguly_p = [
    (5, r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (5, r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (8, r"\b(matka|ojciec|rodzina)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (0, r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),   # fallback
    # TODO (Zadanie 5): dodaj własne reguły z sensownymi priorytetami
]

def odpowiedz_p(wypowiedz):
    w = normalizuj(wypowiedz)
    trafienia = []
    for priorytet, wzorzec, szablony in reguly_p:
        m = re.search(wzorzec, w)
        if m:
            trafienia.append((priorytet, m, szablony))
    # TODO (Zadanie 5): wybierz trafienie o najwyższym priorytecie,
    #                   odbij grupy, wstaw do losowego szablonu i zwróć.
    ________   # <- uzupełnij

In [ ]:
# Test Zadania 5: dwa wyzwalacze („czuję się" i „rodzina") — powinna wygrać rodzina (8 > 5).
print(odpowiedz_p("Czuję się źle, bo moja rodzina mnie nie rozumie."))

## 8. Część B — złam ELIZĘ

Teraz pracujesz jak adwersarz: **celowo** szukasz wypowiedzi,
które obnażają brak rozumienia.

### 🔧 Zadanie 6 (kod + odpowiedź pisemna)
Znajdź **3–4 różne typy** sytuacji, w których iluzja zawodzi.
Dla każdej: podaj konkretne zdanie, pokaż odpowiedź ELIZY i wyjaśnij,
**dlaczego** mechanizm tu zawodzi (odwołaj się do tego, jak działa kod).

Gdzie szukać:
- negacja i przeczenie,
- odwołanie do czegoś sprzed kilku zdań (brak pamięci),
- ironia, przenośnia, żart,
- miejsca, gdzie **odbicie zaimków** daje niegramatyczny polski,
- miejsca, gdzie wycięty fragment nie pasuje gramatycznie do szablonu.

In [ ]:
przyklady_lamiace = [
    # TODO (Zadanie 6): wstaw tu swoje zdania łamiące
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz(zdanie)}\n")

**Twoja analiza (Zadanie 6):**

1. Typ sytuacji: … — przykład: … — dlaczego zawodzi: …
2. …
3. …

## 9. Refleksja

### 🔧 Zadanie 7 (krótko, 3–5 zdań na punkt)
1. Dlaczego wybór roli **terapeuty rogeriańskiego** był dla ELIZY sprytny?
   Czy Twoja ELIZA działałaby równie przekonująco jako np. przewodnik turystyczny?
2. Po zbudowaniu i złamaniu ELIZY: skąd bierze się **efekt ELIZY**?
   Gdzie — w programie czy w rozmówcy — „powstaje" wrażenie rozumienia?

**Twoje odpowiedzi (Zadanie 7):**

1. …
2. …

## Na następnych zajęciach

Puścimy **te same** zdania przez Twoją ELIZĘ i przez duży model językowy (LLM)
i sprawdzimy: co robi LLM, czego ELIZA nie może — i czy w obu przypadkach
wrażenie rozumienia ma to samo źródło. **Zachowaj ten notebook.**
